[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/thin_lens/thin_lens.ipynb)

# Thin Lenses as Maps on Rays

This example traces rays through thin lenses, a prism and a mirror. A thin lens bends every ray that crosses it by an amount set by how far from the lens's centre it crosses: rays from one point leave it aimed at one image point, and parallel rays leave it aimed at its focus. Stack several elements and the whole train still sends the rays from one point through one image, wherever the elements stand and however they are tilted.

A ray is a line of the plane, the join of two points, and a thin lens at the origin is a map on lines: `Line - (Line & origin) * (mv.x / focal)`, each line sheared by its incidence with the lens's centre. A lens elsewhere is that map carried by a motor, `motor >> lens(motor << Line)`, and a train of elements is the composition of their maps, one map acting on the whole bundle. Lines are global, so the free space between elements needs no map of its own, and an image is the meet of two rays that leave the train.

In ray-transfer optics a ray reads as its height and slope where it crosses one reference plane, each element as an ABCD matrix on that pair, and the free space between elements as a propagation matrix of its own.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from functools import reduce
from itertools import accumulate

import numpy as np

from numga import stack
from examples.animation import save_figure, show_animation
from examples.optics.thin_lens import render, scenarios
from examples.optics.thin_lens.core import ELEMENTS, Crossing, Line, LineMap, Motor, Plane, Point, home, mv, origin

np.set_printoptions(precision=4, suppress=True)

## 1. One lens, and two

A lens anywhere is the lens at the origin, carried onto its plane by the translator that moves the origin there. Rays from an object point through the lens all pass through one point after it, the image, found as the meet of any two of them; its distance from the lens obeys the thin lens equation. Two lenses compose into one map, and parallel rays through both meet at the system's back focal point, where Gullstrand's formula puts it.

In [ ]:
# Where each lens stands along x and its focal length; the object; the pupil's half-height, in rays.
first_lens, first_focal = 1.5, 1.0
second_lens, second_focal = 2.1, 0.5
subject = mv.point([-2.0, 0.5])
pupil_height, n_rays = 0.8, 9

In [ ]:
def thin_lens(plane: Plane, focal: float) -> LineMap:
    """A thin lens in a vertical plane, as a map on lines.

    At the origin it shears a line's x coefficient by the line's incidence with the origin over
    the focal length. Anywhere else it is that map conjugated by the translator onto its plane,
    generated by the ideal point along y scaled by the plane's offset from the origin.
    """
    shear: LineMap = Line - (Line & origin) * (mv.x / focal)
    shift: Motor = (mv.wx * (plane & origin) * 0.5).exp()
    return shift >> shear(shift << Line)


first_plane, second_plane = mv.x - mv.w * first_lens, mv.x - mv.w * second_lens   # [] Plane each
first, second = thin_lens(first_plane, first_focal), thin_lens(second_plane, second_focal)   # [] Line <- Line each
# Rays from the object through a pupil of points on the first lens's plane, about the optical axis.
axis = origin & mv.direction([1, 0])                                           # [] Line
pupil = (first_plane ^ axis) + mv.direction([0, 1]) * np.linspace(-pupil_height, pupil_height, n_rays)   # [rays] Point
rays = subject & pupil                                                         # [rays] Line
out = first(rays)                                                              # [rays] Line
image = out[0] ^ out[1]                                                        # [] Crossing
# Two lenses: one map. Parallel rays, joins with the point at infinity along x, meet after it at the focus.
system = second(first)                                                         # [] Line <- Line
parallel = mv.direction([1, 0]) & pupil                                        # [rays] Line
focused = system(parallel)                                                     # [rays] Line
focus = focused[0] ^ focused[1]                                                # [] Crossing
one_lens, _, two_lenses, _, _, _ = scenarios.lenses()
save_figure(render.draw_lenses(one_lens, [first_plane], two_lenses, [first_plane, second_plane]), "thin_lens");

In [ ]:
# Signed distances from the first lens: a point's incidence with the lens plane over its weight.
object_distance = -(first_plane & subject) / (mv.w & subject)                  # [] Scalar
image_distance = (first_plane & image) / (mv.w & image)                        # [] Scalar
print("1 / object distance + 1 / image distance:", (1 / object_distance + 1 / image_distance).to_array(), " 1 / focal:", 1 / first_focal)
gap = (first_plane - second_plane) & origin
effective = 1 / (1 / first_focal + 1 / second_focal - gap / (first_focal * second_focal))
print("focus behind the second lens:", ((second_plane & focus) / (mv.w & focus)).to_array(),
      " Gullstrand:", (effective * (first_focal - gap) / first_focal).to_array())

## 2. A train of elements, moving

Every element is a map on lines defined in the plane x = 0: a thin lens shears a line by its incidence with the centre, a thin prism by its incidence with a point at infinity, the same turn at every height, and a flat mirror is the sandwich by its plane. A motor places each, and the train is their composition, built element by element as the bundle passes through. The first lens slides and tilts and the mirror rocks; a tilted ideal lens still images, since concurrent rays stay concurrent, and the image, the meet of two outgoing rays, moves with them.

In [ ]:
# The train's subject and the point its fan of rays aims at, the fan's half-height in rays; where each
# element stands along x; how far the first lens slides and tilts and the mirror rocks, about its tilt at
# rest; the frames, and their duration in milliseconds.
train_subject = mv.point([-1.0, 0.5])
aim, fan_height, n_fan_rays = mv.point([1.0, 0.0]), 0.6, 7
stations = np.array([1.0, 1.9, 2.2, 3.2])
slide, lens_tilt, rock, mirror_tilt = 0.3, 0.3, 0.1, np.pi / 4
n_frames, frame_duration = 72, 60

In [ ]:
def trace(rays: Line, motors: tuple[Motor, ...], elements: tuple[LineMap, ...]):
    """Place each element by its motor and pass the bundle through the train.

    Returns the element planes [element], the bundle before and after each element
    [element + 1, ray], and the train: the composition of the placed elements, one map on
    lines that acts on the whole bundle.
    """
    # Each element conjugated into place.
    placed = [motor >> element(motor << Line) for motor, element in zip(motors, elements)]   # [] Line <- Line each
    # The bundle after each element, and the train as one map, each placed element applied to the last.
    legs = stack(list(accumulate(placed, lambda bundle, element: element(bundle), initial=rays)))   # [element + 1, ray] Line
    train = reduce(lambda composed, element: element(composed), placed, Line)             # [] Line <- Line
    return stack([motor >> home for motor in motors]), legs, train


def placements(phase: float) -> tuple[Motor, ...]:
    """The four elements' motors at a phase of the loop: the first lens slides and tilts, the prism and
    the second lens stay, and the mirror rocks about its pivot."""
    return ((mv.wx * (-(stations[0] + slide * np.sin(phase)) / 2)).exp() * (mv.xy * (lens_tilt * np.sin(2 * phase) / 2)).exp(),
            (mv.wx * (-stations[1] / 2)).exp(),
            (mv.wx * (-stations[2] / 2)).exp(),
            (mv.wx * (-stations[3] / 2)).exp() * (mv.xy * ((mirror_tilt + rock * np.cos(phase)) / 2)).exp())


def frames():
    """Per frame the subject, the element planes, the bundle before and after each element, the train as
    one map, and the image, the meet of two outgoing rays."""
    for phase in np.linspace(0.0, 2 * np.pi, n_frames, endpoint=False):
        planes, legs, composed = trace(fan, placements(phase), ELEMENTS)
        back = composed(fan)                                                   # [rays] Line
        yield train_subject, planes, legs, composed, back[0] ^ back[-1]


fan = train_subject & (aim + mv.direction([0, 1]) * np.linspace(-fan_height, fan_height, n_fan_rays))   # [rays] Line
states = list(frames())
show_animation(render.animate_train(states), "thin_lens", frame_duration)

In [ ]:
# Every ray out of the train passes through the image, in every frame.
misses = max(np.abs((composed(legs[0]) ^ picture).kernel).max() for _, _, legs, composed, picture in states)
print("largest distance of an outgoing ray from the image:", misses)

In [ ]:
# checks
# Every ray through one lens passes through the image, at the distance the thin lens equation gives; two
# lenses focus parallel rays on the axis where Gullstrand's formula puts it; the train images in every frame.
np.testing.assert_allclose((out ^ image).kernel, 0.0, atol=1e-11)
np.testing.assert_allclose((1 / object_distance + 1 / image_distance).to_array(), 1 / first_focal, atol=1e-12)
np.testing.assert_allclose(((second_plane & focus) / (mv.w & focus)).to_array(), (effective * (first_focal - gap) / first_focal).to_array(), atol=1e-12)
np.testing.assert_allclose((mv.y ^ focus).kernel, 0.0, atol=1e-12)
assert misses < 1e-10